# PCFI261 - Solemne 3
## Notebook 01: del video a la trayectoria angular del pendulo

Este notebook guia la extraccion de una trayectoria desde un video propio de un pendulo. La salida principal es un archivo `trayectoria_pendulo.csv` con, al menos, las columnas `frame`, `t`, `x`, `y`, `theta` y `t_rel`.

El flujo reproduce la primera parte de la evaluacion: video -> posicion en pixeles -> angulo con unidades.

> **Uso esperado.** Esta es una plantilla de trabajo. No debe entregarse sin completar los valores marcados como `TODO`, sin revisar las figuras, ni sin discutir las decisiones experimentales y fisicas en el informe.

## 0. Preparacion

Instale, si es necesario:

```bash
pip install opencv-python numpy pandas matplotlib
```

Coloque el video en la misma carpeta que este notebook y ajuste la variable `VIDEO_PATH`. Formatos comunes: `.mp4`, `.mov`, `.avi`.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
# ============================
# CONFIGURACION DEL ESTUDIANTE
# ============================

VIDEO_PATH = Path("pendulo.mp4")       # TODO: nombre real del video
OUTPUT_DIR = Path("salidas_pendulo")
OUTPUT_DIR.mkdir(exist_ok=True)

# Intervalo de video que se analizara. Use None para no recortar.
T_INICIO = None   # ejemplo: 2.0
T_FIN = None      # ejemplo: 12.0

# Segmentacion por color en HSV.
# TODO: ajustar estos limites al color real de la masa.
HSV_LOWER = np.array([0, 80, 80], dtype=np.uint8)
HSV_UPPER = np.array([20, 255, 255], dtype=np.uint8)

# Limpieza minima de la mascara.
MIN_AREA = 20
KERNEL_SIZE = 5

# Pivote aproximado en pixeles. Complete despues de mirar la trayectoria.
PIVOTE_X = None   # ejemplo: 320.0
PIVOTE_Y = None   # ejemplo: 80.0

# Escala opcional. Si no la conoce, deje None y trabaje con angulos.
PIXEL_TO_METER = None  # ejemplo: 0.0012

## 1. Metadatos del video

Reporte estos datos en el informe: FPS, numero de cuadros, duracion y resolucion. Tambien describa longitud aproximada del pendulo, iluminacion, posicion de la camara y fondo usado.


In [ ]:
def leer_metadatos_video(video_path):
    video = cv2.VideoCapture(str(video_path))
    if not video.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el video: {video_path}")

    fps = float(video.get(cv2.CAP_PROP_FPS))
    nframes = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = nframes / fps if fps > 0 else np.nan
    video.release()

    return {
        "fps": fps,
        "nframes": nframes,
        "duration_s": duration,
        "width_px": width,
        "height_px": height,
    }

metadata = leer_metadatos_video(VIDEO_PATH)
metadata

## 2. Visualizacion de cuadros de muestra

Use estos cuadros para decidir si conviene recortar el intervalo, cambiar iluminacion o ajustar los limites HSV.


In [ ]:
def leer_frame(video_path, frame_id):
    video = cv2.VideoCapture(str(video_path))
    video.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ok, frame_bgr = video.read()
    video.release()
    if not ok:
        raise ValueError(f"No se pudo leer el frame {frame_id}")
    return frame_bgr

fps = metadata["fps"]
nframes = metadata["nframes"]

sample_ids = np.linspace(0, max(nframes - 1, 0), 6, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, frame_id in zip(axes.ravel(), sample_ids):
    frame_bgr = leer_frame(VIDEO_PATH, frame_id)
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(frame_rgb)
    ax.set_title(f"frame={frame_id}, t={frame_id/fps:.2f} s")
    ax.axis("off")
fig.tight_layout()

## 3. Calibracion de color HSV

Ajuste `HSV_LOWER` y `HSV_UPPER` hasta que la mascara seleccione principalmente la masa. El objetivo no es seleccionar muchos pixeles, sino seleccionar pixeles confiables de la masa.


In [ ]:
# Cambie este frame para probar la segmentacion.
FRAME_TEST = int(0.5 * nframes)

frame_bgr = leer_frame(VIDEO_PATH, FRAME_TEST)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, HSV_LOWER, HSV_UPPER)

kernel = np.ones((KERNEL_SIZE, KERNEL_SIZE), np.uint8)
mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(frame_rgb)
axes[0].set_title("frame original")
axes[0].axis("off")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("mascara HSV")
axes[1].axis("off")
axes[2].imshow(mask_clean, cmap="gray")
axes[2].set_title("mascara limpia")
axes[2].axis("off")
fig.tight_layout()

## 4. Extraccion de centroide cuadro a cuadro

El metodo usado aqui toma el componente conectado mas grande de la mascara y calcula su centroide. Si su masa no tiene color distinguible, puede reemplazar esta seccion por seleccion manual, umbral de intensidad, seguimiento por template matching u otro procedimiento justificado.


In [ ]:
def centroide_componente_principal(mask, min_area=20):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return np.nan, np.nan, 0.0

    # Se ignora la etiqueta 0 porque corresponde al fondo.
    areas = stats[1:, cv2.CC_STAT_AREA]
    idx = int(np.argmax(areas)) + 1
    area = float(stats[idx, cv2.CC_STAT_AREA])
    if area < min_area:
        return np.nan, np.nan, area

    x, y = centroids[idx]
    return float(x), float(y), area


def extraer_trayectoria(video_path, hsv_lower, hsv_upper, t_inicio=None, t_fin=None,
                        min_area=20, kernel_size=5):
    video = cv2.VideoCapture(str(video_path))
    if not video.isOpened():
        raise FileNotFoundError(f"No se pudo abrir el video: {video_path}")

    fps = float(video.get(cv2.CAP_PROP_FPS))
    rows = []
    frame_id = 0
    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    while True:
        ok, frame_bgr = video.read()
        if not ok:
            break

        t = frame_id / fps
        usar_frame = True
        if t_inicio is not None and t < t_inicio:
            usar_frame = False
        if t_fin is not None and t > t_fin:
            usar_frame = False

        if usar_frame:
            hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
            mask = cv2.inRange(hsv, hsv_lower, hsv_upper)
            if kernel_size is not None and kernel_size > 1:
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
                mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            x, y, area = centroide_componente_principal(mask, min_area=min_area)
            rows.append({
                "frame": frame_id,
                "t": t,
                "x": x,
                "y": y,
                "area_px": area,
                "detectado": np.isfinite(x) and np.isfinite(y),
            })

        frame_id += 1

    video.release()
    return pd.DataFrame(rows)

traj = extraer_trayectoria(
    VIDEO_PATH, HSV_LOWER, HSV_UPPER,
    t_inicio=T_INICIO, t_fin=T_FIN,
    min_area=MIN_AREA, kernel_size=KERNEL_SIZE,
)

print(traj.head())
print("frames analizados:", len(traj))
print("detecciones validas:", int(traj["detectado"].sum()))

## 5. Control de calidad de la trayectoria

Revise si hay puntos perdidos, saltos o detecciones del fondo. En el informe debe explicar como verifico que los puntos detectados corresponden realmente a la masa.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ok = traj["detectado"].to_numpy(dtype=bool)
axes[0].plot(traj.loc[ok, "x"], traj.loc[ok, "y"], ".")
axes[0].invert_yaxis()
axes[0].set_xlabel("x [pixeles]")
axes[0].set_ylabel("y [pixeles]")
axes[0].set_title("trayectoria en el plano del video")

axes[1].plot(traj["t"], traj["x"], ".-", label="x")
axes[1].plot(traj["t"], traj["y"], ".-", label="y")
axes[1].set_xlabel("t [s]")
axes[1].set_ylabel("posicion [pixeles]")
axes[1].legend()
axes[1].set_title("series temporales")

axes[2].plot(traj["t"], traj["area_px"], ".-")
axes[2].set_xlabel("t [s]")
axes[2].set_ylabel("area detectada [pixeles]")
axes[2].set_title("area de la mascara")

fig.tight_layout()

In [ ]:
# Limpieza minima: se descartan frames sin deteccion.
# TODO: si elimina outliers adicionales, justifique el criterio en el informe.

df = traj.loc[traj["detectado"]].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["x", "y", "t"])
df = df.sort_values("t").reset_index(drop=True)

print(df.head())
print(df.tail())

## 6. Pivote, angulo y escala

Complete `PIVOTE_X` y `PIVOTE_Y` usando inspeccion del video, una marca visible, o un ajuste geometrico. La definicion de `theta` debe ser consistente durante todo el reporte.

La celda siguiente usa la convencion del enunciado: `theta = arctan2(x - x0, y0 - y)`. Luego centra la senal alrededor de su mediana para eliminar un offset de convencion. Si usa otra convencion, indiquelo explicitamente.


In [ ]:
if PIVOTE_X is None or PIVOTE_Y is None:
    # Estimacion automatica solo para partir. Debe revisarse visualmente.
    PIVOTE_X = float(df["x"].median())
    PIVOTE_Y = float(df["y"].min() - 0.25 * (df["y"].max() - df["y"].min()))
    print("ADVERTENCIA: pivote estimado automaticamente. Revise y reemplace PIVOTE_X/PIVOTE_Y.")

x0, y0 = float(PIVOTE_X), float(PIVOTE_Y)

theta_raw = np.arctan2(df["x"].to_numpy() - x0, y0 - df["y"].to_numpy())
df["theta"] = np.unwrap(theta_raw)
df["theta"] = df["theta"] - np.nanmedian(df["theta"])
df["t_rel"] = df["t"] - df["t"].min()

if PIXEL_TO_METER is not None:
    df["x_m"] = (df["x"] - x0) * PIXEL_TO_METER
    df["y_m"] = (df["y"] - y0) * PIXEL_TO_METER

print(df[["frame", "t", "t_rel", "x", "y", "theta"]].head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df["x"], df["y"], ".", label="masa")
axes[0].plot([x0], [y0], "x", markersize=10, label="pivote")
axes[0].invert_yaxis()
axes[0].set_xlabel("x [pixeles]")
axes[0].set_ylabel("y [pixeles]")
axes[0].legend()
axes[0].set_title("trayectoria y pivote")

axes[1].plot(df["t_rel"], df["theta"], ".-")
axes[1].set_xlabel("t_rel [s]")
axes[1].set_ylabel("theta [rad]")
axes[1].set_title("senal angular")

fig.tight_layout()

## 7. Guardado de datos y figuras

El CSV sera usado por los notebooks 02 y 03.


In [ ]:
csv_path = OUTPUT_DIR / "trayectoria_pendulo.csv"
df.to_csv(csv_path, index=False)
print(f"Datos guardados en: {csv_path}")

fig_path = OUTPUT_DIR / "trayectoria_y_theta.png"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df["x"], df["y"], ".")
axes[0].plot([x0], [y0], "x", markersize=10)
axes[0].invert_yaxis()
axes[0].set_xlabel("x [pixeles]")
axes[0].set_ylabel("y [pixeles]")
axes[0].set_title("trayectoria")
axes[1].plot(df["t_rel"], df["theta"], ".-")
axes[1].set_xlabel("t_rel [s]")
axes[1].set_ylabel("theta [rad]")
axes[1].set_title("theta(t)")
fig.tight_layout()
fig.savefig(fig_path, dpi=200)
print(f"Figura guardada en: {fig_path}")

## 8. Elementos que deben aparecer en el informe

- Descripcion del montaje experimental.
- FPS, duracion y resolucion del video.
- Metodo de segmentacion o seguimiento.
- Justificacion del pivote y de la variable angular.
- Criterios de limpieza de datos.
- Archivo CSV reproducible y figuras finales.
